In [ ]:
!pip install pandas scikit-learn pyarrow

In [ ]:
import pandas as pd
import os
import time

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
AUTOR_EXPERIMENTO = "Oscar Penuela"

# Ganadores de las fases anteriores
WINNING_SPACY_FOLDER = "Exp04_Solo_Elongacion"
WINNING_VECTORIZER = "TFIDF"
WINNING_BIGRAMS = True

# Ruta local a los parquet procesados
DATA_PATH = "./data/train_parquet"  # <- Ajustar a tu ruta local

TEXT_COLUMN = "ablation_text"
TARGET_COLUMN = "label"
MAX_FEATURES = 5000

In [ ]:
from sklearn.naive_bayes import MultinomialNB
MODELO_ELEGIDO = "MultinomialNB"
clf = MultinomialNB()

print(f"Modelo seleccionado y listo para entrenar: {MODELO_ELEGIDO}")

In [ ]:
def load_parquet_local(path: str) -> pd.DataFrame:
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.parquet')]
    df_list = [pd.read_parquet(f) for f in files]
    return pd.concat(df_list, ignore_index=True)

print(f"Cargando dataset: {WINNING_SPACY_FOLDER}...")
full_train_df = load_parquet_local(DATA_PATH)

print(f"Filas cargadas: {len(full_train_df)}")
print("Columnas disponibles:", full_train_df.columns.tolist())

In [ ]:
X = full_train_df[TEXT_COLUMN].fillna("")
y = full_train_df[TARGET_COLUMN]

print("Dividiendo en Entrenamiento (80%) y Validación (20%)...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Val: {len(X_val)}")

In [ ]:
# Configurar el vectorizador de la Fase B
ngram_range = (1, 2) if WINNING_BIGRAMS else (1, 1)

if WINNING_VECTORIZER == "TFIDF":
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)
else:
    vectorizer = CountVectorizer(max_features=MAX_FEATURES, ngram_range=ngram_range)

# Armar el Pipeline
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("model", clf)
])

print(f"Pipeline ensamblado: {WINNING_VECTORIZER} + {MODELO_ELEGIDO}")

In [ ]:
print(f"Entrenando {MODELO_ELEGIDO}...")
start_time = time.time()

pipeline.fit(X_train, y_train)

elapsed = time.time() - start_time
print(f"Entrenamiento completado en {elapsed:.2f} segundos.")

y_pred = pipeline.predict(X_val)
acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="macro")

print(f"\nResultados -> Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}")
print(f"\nReporte de Clasificación:")
print(classification_report(y_val, y_pred))

In [ ]:
import json
from datetime import datetime

model_card = {
    "model_name": f"Modelo_{MODELO_ELEGIDO}",
    "description": f"Algoritmo: {MODELO_ELEGIDO}. Vectorizador: {WINNING_VECTORIZER}. NLP: {WINNING_SPACY_FOLDER}.",
    "author": AUTOR_EXPERIMENTO,
    "dataset_version": WINNING_SPACY_FOLDER,
    "version": str(datetime.now()),
    "performance": {"validation_accuracy": round(acc, 4), "validation_f1_macro": round(f1, 4)},
    "intended_use": "Clasificación de sentimientos."
}

print(json.dumps(model_card, indent=4))